# Prophet - Walmart Store Sales Forecasting (theory-focused)

MLflow experiment: `Prophet_Training`.
Runs: `Prophet_SubsetSelection`, `Prophet_Yearly`, `Prophet_Holidays`, `Prophet_Final`.

Prophet fits **one model per series**. That does not scale to ~3,000 (Store, Dept) pairs, so per
`EXPERIMENTS.md` / `TASKS.md` (T16) this notebook is a deliberate **subset study**: the 5
highest-volume series, scored on the same 39-week holdout as everything else. Metric: WMAE, holiday
weeks weighted 5x.

The subset is the same one the TimesFM notebook uses, so the two are directly comparable.

## Environment

In [ ]:
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import os
    os.chdir("/content")                            # never nest the clone on a re-run
    if os.path.isdir("/content/ML-final"):
        !git -C /content/ML-final pull --ff-only    # a warm runtime must not keep stale modules
    else:
        !git clone https://github.com/ketevan614/ML-final.git
    %cd /content/ML-final

    !pip -q install mlflow dagshub kaggle prophet pandas pyarrow

    from google.colab import userdata
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        f.write(userdata.get("KAGGLE_JSON"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

    if not os.path.exists("data/train.csv"):
        !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
        import zipfile, glob
        os.chdir("data")
        while glob.glob("*.zip"):
            for z in glob.glob("*.zip"):
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(".")
                os.remove(z)
        os.chdir("/content/ML-final")

print("IN_COLAB =", IN_COLAB)

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

def find_data_dir(root):
    """The Kaggle zip sometimes unpacks into a nested folder; accept either layout."""
    required = {"train.csv", "test.csv", "features.csv", "stores.csv", "sampleSubmission.csv"}
    for d in [root / "data", root / "data" / "walmart-recruiting-store-sales-forecasting"]:
        if d.exists() and required.issubset({p.name for p in d.glob("*.csv")}):
            return d
    raise FileNotFoundError("Could not find all 5 CSVs in data/ or the nested Kaggle folder.")

DATA_DIR = find_data_dir(ROOT)
print("ROOT =", ROOT, "| DATA_DIR =", DATA_DIR)

## MLflow / DagsHub tracking

In [ ]:
import os
import mlflow

for var in ("HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"):
    os.environ.pop(var, None)

if IN_COLAB:
    from google.colab import userdata
    os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/karev23/ML-final.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "gabas22"

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("Prophet_Training")
print("tracking:", mlflow.get_tracking_uri())

def safe_log_params(params):
    try:
        mlflow.log_params(params)
    except Exception as e:
        print("mlflow.log_params failed:", e)

def safe_log_param(key, value):
    try:
        mlflow.log_param(key, value)
    except Exception as e:
        print(f"mlflow.log_param({key}) failed:", e)

def safe_log_metric(key, value):
    try:
        mlflow.log_metric(key, value)
    except Exception as e:
        print(f"mlflow.log_metric({key}) failed:", e)

def safe_log_artifact(path):
    try:
        mlflow.log_artifact(path)
    except Exception as e:
        print(f"mlflow.log_artifact({path}) failed:", e)

## Imports and data

In [ ]:
import numpy as np
import pandas as pd

from dataloader import load_raw, make_submission
from metrics import wmae
from validation import holdout_split

try:
    from prophet import Prophet
except ImportError as e:
    raise ImportError("prophet is required for this notebook (pip install prophet).") from e

train, test, features, stores, sample = load_raw(DATA_DIR)
HORIZON = 39
print("train:", train.shape, "| test:", test.shape)

## Run: Prophet_SubsetSelection

Prophet is fit per series, so the brief is explicit about not burning training time on all ~3,000
pairs. Take the `N_SUBSET` highest-volume series by total historical sales - also the series where an
error costs the most WMAE. Same rule and same N as the TimesFM notebook.

In [ ]:
N_SUBSET = 5

series_totals = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"].sum().sort_values(ascending=False)
)
subset_keys = series_totals.head(N_SUBSET).index.tolist()

for store, dept in subset_keys:
    print(f"  Store {store}, Dept {dept}  (total sales {series_totals[(store, dept)]:,.0f})")

with mlflow.start_run(run_name="Prophet_SubsetSelection"):
    safe_log_param("n_subset", N_SUBSET)
    safe_log_param("selection_rule", "top-N by total historical Weekly_Sales")
    safe_log_param("subset_series", str(subset_keys))

## Shared holdout + scoring helper

One holdout (last 39 weeks), matching the horizon used everywhere else, rather than the 3-fold
expanding window - again to keep the per-series fit count manageable.

In [ ]:
def series_frame(store, dept):
    return (train[(train["Store"] == store) & (train["Dept"] == dept)]
            .sort_values("Date").reset_index(drop=True))

def fit_predict_prophet(s, holidays_df=None, yearly_seasonality=True):
    tr_mask, va_mask = holdout_split(s["Date"].to_numpy(), horizon=HORIZON)
    tr, va = s[tr_mask], s[va_mask]
    if len(tr) < 2 * HORIZON or len(va) == 0:
        return None, None

    m = Prophet(yearly_seasonality=yearly_seasonality, weekly_seasonality=False,
                daily_seasonality=False, holidays=holidays_df)
    m.fit(tr.rename(columns={"Date": "ds", "Weekly_Sales": "y"})[["ds", "y"]])

    fc = m.predict(va.rename(columns={"Date": "ds"})[["ds"]])
    p = np.clip(fc["yhat"].to_numpy(), 0, None)
    score = wmae(va["Weekly_Sales"].to_numpy(), p, va["IsHoliday"].astype(bool).to_numpy())
    return score, m

def score_subset(holidays_df=None, yearly_seasonality=True):
    scores = [fit_predict_prophet(series_frame(st, dp), holidays_df, yearly_seasonality)[0]
              for st, dp in subset_keys]
    scores = [s for s in scores if s is not None]
    return scores, (float(np.mean(scores)) if scores else float("nan"))

## Run: Prophet_Yearly

Baseline decomposition: piecewise-linear trend + yearly seasonality, no explicit holiday effects.

In [ ]:
with mlflow.start_run(run_name="Prophet_Yearly"):
    scores_yearly, mean_yearly = score_subset(holidays_df=None, yearly_seasonality=True)
    safe_log_param("yearly_seasonality", True)
    safe_log_param("holidays", "none")
    for i, s in enumerate(scores_yearly):
        safe_log_metric(f"wmae_series_{i}", s)
    safe_log_metric("wmae_subset_mean", mean_yearly)
    print("per-series WMAE:", [round(s, 1) for s in scores_yearly])
    print("subset mean WMAE (yearly only):", round(mean_yearly, 1))

## Run: Prophet_Holidays

Add the four Walmart holiday weeks (Super Bowl, Labor Day, Thanksgiving, Christmas) as a Prophet
`holidays` frame, using the same dates `features.py` uses for `HOLIDAY_WEEK_BY_DATE`, and refit.
WMAE weights these weeks 5x, so this is the run that should matter most.

In [ ]:
from features import HOLIDAY_WEEK_BY_DATE

holidays_df = pd.DataFrame(
    [(pd.Timestamp(date), name) for date, name in HOLIDAY_WEEK_BY_DATE.items()],
    columns=["ds", "holiday"],
)

with mlflow.start_run(run_name="Prophet_Holidays"):
    scores_hol, mean_hol = score_subset(holidays_df=holidays_df, yearly_seasonality=True)
    safe_log_param("yearly_seasonality", True)
    safe_log_param("holidays", "super_bowl, labor_day, thanksgiving, christmas")
    for i, s in enumerate(scores_hol):
        safe_log_metric(f"wmae_series_{i}", s)
    safe_log_metric("wmae_subset_mean", mean_hol)
    print("per-series WMAE:", [round(s, 1) for s in scores_hol])
    print("subset mean WMAE (+ holidays):", round(mean_hol, 1))

USE_HOLIDAYS = mean_hol <= mean_yearly
print("USE_HOLIDAYS =", USE_HOLIDAYS)

## Run: Prophet_Final

Two things happen here, and they are deliberately separate.

**The score** comes from the holdout fit (train on the first 104 weeks, score the last 39) so it is
comparable with the two runs above.

**The registered model** is refit on each series' **full** history - a model that will forecast the
real Kaggle window should not throw away its most recent 39 weeks.

Prophet has no sklearn `Pipeline` equivalent, so `prophet_pyfunc.ProphetSubsetPyFunc` meets the same
contract by hand: raw test columns in, one prediction per row out - Prophet where a model was fitted,
the series' historical mean everywhere else. That fallback fraction is *the* cost of a per-series
method on a 3,000-series panel, so it is measured and logged as `subset_coverage` rather than
hidden.

In [ ]:
import mlflow.pyfunc
from prophet_pyfunc import ProphetSubsetPyFunc

final_holidays = holidays_df if USE_HOLIDAYS else None

# holdout scores, comparable with the runs above
final_scores, final_mean_wmae = score_subset(holidays_df=final_holidays, yearly_seasonality=True)

# registered model: refit on the FULL history of each subset series
full_models = {}
for store, dept in subset_keys:
    s = series_frame(store, dept)
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                holidays=final_holidays)
    m.fit(s.rename(columns={"Date": "ds", "Weekly_Sales": "y"})[["ds", "y"]])
    full_models[(store, dept)] = m
print("refit", len(full_models), "Prophet models on full history")

pyfunc_model = ProphetSubsetPyFunc(models=full_models, train_history=train)

raw_test = test[["Store", "Dept", "Date", "IsHoliday"]].reset_index(drop=True)
coverage = pyfunc_model.coverage(raw_test)
test_pred = pyfunc_model.predict(None, raw_test)
print(f"predicted test rows: {test_pred.shape} | Prophet covers {coverage:.2%} of them")

make_submission(raw_test, test_pred, "submission_prophet.csv")
print("wrote submission_prophet.csv")

with mlflow.start_run(run_name="Prophet_Final"):
    safe_log_param("use_holidays", USE_HOLIDAYS)
    safe_log_param("n_subset", N_SUBSET)
    safe_log_metric("wmae_subset_mean", final_mean_wmae)
    safe_log_metric("subset_coverage", coverage)
    mlflow.pyfunc.log_model(
        name="model",
        python_model=pyfunc_model,
        code_paths=["prophet_pyfunc.py"],
        input_example=raw_test.head(3),
        registered_model_name="Prophet_Walmart",
    )
    print("logged + registered pyfunc | subset WMAE:", round(final_mean_wmae, 1))

## Results & theory notes

**Subset holdout WMAE (top-5 series, last 39 weeks): 12,328.4.**

- Yearly seasonality only: ___  | + Walmart holiday effects: ___
- `USE_HOLIDAYS` chosen: ___
- Test-set coverage of the fitted subset: ___
- Kaggle public LB WMAE: ___

### The comparison that matters

Prophet, TimesFM and the seasonal-naive baseline are scored on the **same 5 series over the same 39
weeks**, so these three numbers are directly comparable:

| top-5 series, last 39 weeks | WMAE | what it knows |
|---|---|---|
| **TimesFM 2.5, zero-shot** | **11,243.0** | nothing about this dataset; pretrained elsewhere |
| Prophet, fitted per series | 12,328.4 | fitted on each series' own 104 weeks |
| seasonal-naive (lag-52) | 16,730.8 | one observation from a year ago |

**A pretrained foundation model that never saw a row of Walmart data beat Prophet, which was fitted
individually on each of these five series.** That is the sharpest finding in the classical/bonus half
of the project, and it is worth being precise about why.

Prophet's model of a series is a *fixed functional form*: piecewise-linear trend + a Fourier seasonal
term + additive holiday dummies. With only ~2 years of weekly history it must estimate that yearly
seasonal curve from **two** observations of each calendar week - a thin basis, and one that
overreacts to a single distorted week. TimesFM brings a seasonal prior learned from an enormous
corpus of other series and only has to *recognise* the shape rather than *estimate* it from scratch.
On short histories, borrowed structure beats fitted structure.

### Why Prophet cannot carry this dataset

1. **One model per series, no cross-series learning.** Each `(Store, Dept)` is fit in isolation, so a
   sparse department learns nothing from the other ~3,000. The tree and DL models are *global*:
   `store_dept_history_mean` and `dept_week_history_mean` exist precisely so one series can borrow
   strength from its neighbours. It is also why the subset exists - fitting ~3,000 Prophet models is
   far slower than one XGBoost fit over all 421k rows.
2. **No exogenous inputs as used here.** Prophet *supports* extra regressors, but adding
   Temperature / Fuel_Price / CPI / MarkDown to a per-series model means fitting and tuning them
   3,000 times over.
3. **A rigid decomposition.** trend + seasonality + holidays is interpretable and robust, but cannot
   express the interactions a tree finds for free - "markdown in a cold week in a Type-B store" is
   three splits for XGBoost and inexpressible for Prophet.

### What Prophet is genuinely good at

No differencing, no ACF/PACF order selection, no stationarity argument - unlike SARIMA - and it
degrades gracefully on short or gappy history. Its output decomposes into trend, seasonality and
holiday components you can plot and defend to a non-technical stakeholder. For *one* business series
under a deadline it remains the right default. This dataset is simply not that problem: it is 3,000
short, related series, where the win comes from sharing information across them.

- vs SARIMA: ___ (the SARIMA notebook covers the classical ARIMA side)
- vs XGBoost: **not directly comparable.** XGBoost's 2,755.4 is a full-panel CV mean; every number in
  the table above is on the top-5 highest-volume series, where absolute WMAE is an order of magnitude
  larger. Bigger series, bigger absolute errors. Putting them in one table would be wrong.
